In [17]:
# ======================================================
# Core libraries for respiration + social behavior analysis
# ======================================================

# --- Core Python ---
import os
import h5py
import re

# --- Numerical & data analysis ---
import numpy as np
import pandas as pd

# --- Plotting ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Signal processing ---
from scipy.signal import butter, filtfilt, resample_poly, find_peaks

# --- Statistics ---
from scipy.stats import wilcoxon

# --- Machine learning & metrics ---
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline



# --- Specialized neurophysiology tools ---
import neurokit2 as nk

# --- Pandas display settings ---
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# ======================================================
# Define file paths and rank map
# ======================================================

# --- Define social rank map for all subjects ---
rank_map = {
    "1_1": "Subordinate",
    "1_2": "Dominant",
    "2_3": "Subordinate",
    "2_4": "Dominant",
    "3_5": "Subordinate",
    "3_6": "Dominant",
    "4_7": "Subordinate",
    "4_8": "Dominant"
}

# ======================================================
# Respiration + BORIS paths (valence and cagemate sets)
# ======================================================

# --- Valence (RI1, RI2, BLRI) ---
resp_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_6_p5_3_nRB3_20250621_125312_merged.h5",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_6_p5_3_nRB3_20250621_131158_merged.h5",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_7_p5_2_nRB3_20250621_150707_merged.h5",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_7_p5_2_nRB3_20250621_152519_merged.h5",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_3_p5_3_nRB3_20250622_104059_merged (1).h5",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_3_p5_3_nRB3_20250622_110216_merged.h5",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_8_p5_1_nRB3_20250621_165214_merged.h5",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_8_p5_1_nRB3_20250621_171318_merged.h5",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_1_p5_2_nRB6_20250622_143958_merged.h5",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_1_p5_2_nRB6_20250622_150457_merged.h5",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_2_p5_1_nRB6_20250622_170742_merged.h5",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_2_p5_1_nRB6_20250622_173049_merged.h5",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_4_p5_4_nRB3_20250622_123424_merged.h5",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_4_p5_4_nRB3_20250622_125648_merged.h5",
    "RI1_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_5_p5_4_nRB3_20250621_105014_merged.h5",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_5_p5_4_nRB3_20250621_112618_merged.h5",
    # Baseline recordings (pre-valence)
    "BLRI_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_1_p5_2_nRB6_20250622_141846_merged.h5",
    "BLRI_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_2_p5_1_nRB6_20250622_164833_merged.h5",
    "BLRI_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_3_p5_3_nRB3_20250622_101813_merged.h5",
    "BLRI_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_4_p5_4_nRB3_20250622_121708_merged.h5",
    "BLRI_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s3_6_p5_3_nRB3_20250621_123634_merged.h5",
    "BLRI_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_7_p5_2_nRB3_20250621_144806_merged.h5",
    "BLRI_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_8_p5_1_nRB3_20250621_163506_merged.h5",
}

boris_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s3_6_p5_3_nRB3_HEEPS.csv",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_6_p_5_3_nRB3_2025062.csv",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_3_p5_3_nRB3_20250622_104059.1.csv",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_3_p5_3_nRB3_20250622_1102116.1.csv",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_8_p5_1_nRB3_HEEPS.csv",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_8_p5_1_nRB3_20250621_HEEPS.csv",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_1_p5_2_nRB6_20250622_143958.1.csv",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_1_p5_2_nRB6_20250622_150457.1.csv",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_2_p5_1_nRB6_20250622_170742.1.csv",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_2_p5_1_nRB6_20250622_173049.1.csv",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_4_p5_4_nRB3_20250622_123424.1.csv",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_4_p5_4_nRB3_20250622_125648.1.csv",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_5_p5_4_nRB3_20250621.csv",
}

# --- Cagemate interaction (CM) ---

# --- Respiration file paths (.h5) --- # Baseline recordings
resp_paths_bl = {
    "BL_1_1_d1_2": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s1_1_d1_2_20250623_103713_merged.h5",
    "BL_1_2_sub1_1": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s1_2_sub_1_1_20250623_120135_merged.h5",
    "BL_2_3_d2_4": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s2_3_d2_4_20250623_145448_merged.h5",
    "BL_2_4_sub2_3": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s2_4_sub2_3_20250623_141419_merged.h5",
    "BL_3_5_d3_6": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s3_5_d3_6_20250623_154154_merged.h5",
    "BL_3_6_sub3_5": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s3_6_sub3_5_20250623_172635_merged.h5",
    "BL_4_7_d4_8": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s4_7_d4_8_20250623_185042_merged.h5",
    "BL_4_8_sub4_7": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s4_8_sub4_7_20250623_180810_merged.h5"
}
resp_paths_cm = {
    "CM_1_1_d1_2": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_1_d1_2_20250623_111352_merged.h5",
    "CM_1_2_sub1_1": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_2_sub1_1_20250623_133932_merged.h5",
    "CM_2_3_d2_4": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_3_d2_4_20250623_151153_merged.h5",
    "CM_2_4_sub2_3": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_4_sub2_3_20250623_143348_merged.h5",
    "CM_3_5_d3_6": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_5_d3_6_20250623_170708_merged.h5",
    "CM_3_6_sub3_5": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_6_sub3_5_20250623_174348_merged.h5",
    "CM_4_7_d4_8": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_7_d4_8_20250623_193718_merged.h5",
    "CM_4_8_sub4_7": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_8_sub4_7_20250623_182649_merged.h5",
}

boris_paths_cm = {
    "CM_1_1_d1_2": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s1_1_d1_2_20250623_111352.1.csv",
    "CM_1_2_sub1_1": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s1_2_sub1_1_20250623_133932.1_VT.csv",
    "CM_2_3_d2_4": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s2_3_d2_4_20250623_151153.1.csv",
    "CM_2_4_sub2_3": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s2_4_sub2_3_20250623_143348.1_VT.csv",
    "CM_3_5_d3_6": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s3_5_d3_6_20250623_160001.csv",
    "CM_3_6_sub3_5": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s3_6_sub3_5_20250623_174348.csv",
    "CM_4_7_d4_8": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s4_7_d4_8_20250623_193718.csv",
    "CM_4_8_sub4_7": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s4_8_sub4_7_20250623_182649.1_VT.csv",
}

# ======================================================
# Summary check
# ======================================================

def summarize_paths(resp_dict, boris_dict, label):
    resp_set = set(resp_dict.keys())
    boris_set = set(boris_dict.keys())
    missing = resp_set - boris_set
    print(f"\n {label} summary")
    print(f"Resp files: {len(resp_set)} | BORIS files: {len(boris_set)} | No (non-social) BORIS: {len(missing)}")
    if missing:
        print("Missing trials:", ", ".join(sorted(missing)))

summarize_paths(resp_paths, boris_paths, "Valence (RI1/RI2/BLRI)")
summarize_paths(resp_paths_cm, boris_paths_cm, "Cagemate (CM)")



 Valence (RI1/RI2/BLRI) summary
Resp files: 23 | BORIS files: 15 | No (non-social) BORIS: 8
Missing trials: BLRI_1_1, BLRI_1_2, BLRI_2_3, BLRI_2_4, BLRI_3_6, BLRI_4_7, BLRI_4_8, RI1_3_5

 Cagemate (CM) summary
Resp files: 8 | BORIS files: 8 | No (non-social) BORIS: 0


In [3]:
# ======================================================
# Define file paths and rank map
# ======================================================

# --- Define social rank map for all subjects ---
rank_map = {
    "1_1": "Subordinate",
    "1_2": "Dominant",
    "2_3": "Subordinate",
    "2_4": "Dominant",
    "3_5": "Subordinate",
    "3_6": "Dominant",
    "4_7": "Subordinate",
    "4_8": "Dominant"
}

# ======================================================
# Respiration + BORIS paths (valence and cagemate sets)
# ======================================================

# --- Valence (RI1, RI2, BLRI) ---
resp_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_6_p5_3_nRB3_20250621_125312_merged.h5",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_6_p5_3_nRB3_20250621_131158_merged.h5",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_7_p5_2_nRB3_20250621_150707_merged.h5",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_7_p5_2_nRB3_20250621_152519_merged.h5",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_3_p5_3_nRB3_20250622_104059_merged (1).h5",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_3_p5_3_nRB3_20250622_110216_merged.h5",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_8_p5_1_nRB3_20250621_165214_merged.h5",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_8_p5_1_nRB3_20250621_171318_merged.h5",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_1_p5_2_nRB6_20250622_143958_merged.h5",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_1_p5_2_nRB6_20250622_150457_merged.h5",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_2_p5_1_nRB6_20250622_170742_merged.h5",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_2_p5_1_nRB6_20250622_173049_merged.h5",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_4_p5_4_nRB3_20250622_123424_merged.h5",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_4_p5_4_nRB3_20250622_125648_merged.h5",
    "RI1_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_5_p5_4_nRB3_20250621_105014_merged.h5",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_5_p5_4_nRB3_20250621_112618_merged.h5",
    # Baseline recordings (pre-valence)
    "BLRI_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_1_p5_2_nRB6_20250622_141846_merged.h5",
    "BLRI_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_2_p5_1_nRB6_20250622_164833_merged.h5",
    "BLRI_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_3_p5_3_nRB3_20250622_101813_merged.h5",
    "BLRI_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_4_p5_4_nRB3_20250622_121708_merged.h5",
    "BLRI_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s3_6_p5_3_nRB3_20250621_123634_merged.h5",
    "BLRI_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_7_p5_2_nRB3_20250621_144806_merged.h5",
    "BLRI_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_8_p5_1_nRB3_20250621_163506_merged.h5",
}

boris_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s3_6_p5_3_nRB3_HEEPS.csv",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_6_p_5_3_nRB3_2025062.csv",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_3_p5_3_nRB3_20250622_104059.1.csv",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_3_p5_3_nRB3_20250622_1102116.1.csv",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_8_p5_1_nRB3_HEEPS.csv",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_8_p5_1_nRB3_20250621_HEEPS.csv",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_1_p5_2_nRB6_20250622_143958.1.csv",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_1_p5_2_nRB6_20250622_150457.1.csv",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_2_p5_1_nRB6_20250622_170742.1.csv",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_2_p5_1_nRB6_20250622_173049.1.csv",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_4_p5_4_nRB3_20250622_123424.1.csv",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_4_p5_4_nRB3_20250622_125648.1.csv",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_5_p5_4_nRB3_20250621.csv",
}

# --- Cagemate interaction (CM) and baselines (BL) ---

resp_paths_bl = {
    "BL_1_1_d1_2": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s1_1_d1_2_20250623_103713_merged.h5",
    "BL_1_2_sub1_1": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s1_2_sub_1_1_20250623_120135_merged.h5",
    "BL_2_3_d2_4": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s2_3_d2_4_20250623_145448_merged.h5",
    "BL_2_4_sub2_3": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s2_4_sub2_3_20250623_141419_merged.h5",
    "BL_3_5_d3_6": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s3_5_d3_6_20250623_154154_merged.h5",
    "BL_3_6_sub3_5": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s3_6_sub3_5_20250623_172635_merged.h5",
    "BL_4_7_d4_8": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s4_7_d4_8_20250623_185042_merged.h5",
    "BL_4_8_sub4_7": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s4_8_sub4_7_20250623_180810_merged.h5"
}

resp_paths_cm = {
    "CM_1_1_d1_2": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_1_d1_2_20250623_111352_merged.h5",
    "CM_1_2_sub1_1": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_2_sub1_1_20250623_133932_merged.h5",
    "CM_2_3_d2_4": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_3_d2_4_20250623_151153_merged.h5",
    "CM_2_4_sub2_3": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_4_sub2_3_20250623_143348_merged.h5",
    "CM_3_5_d3_6": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_5_d3_6_20250623_170708_merged.h5",
    "CM_3_6_sub3_5": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_6_sub3_5_20250623_174348_merged.h5",
    "CM_4_7_d4_8": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_7_d4_8_20250623_193718_merged.h5",
    "CM_4_8_sub4_7": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_8_sub4_7_20250623_182649_merged.h5",
}

boris_paths_cm = {
    "CM_1_1_d1_2": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s1_1_d1_2_20250623_111352.1.csv",
    "CM_1_2_sub1_1": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s1_2_sub1_1_20250623_133932.1_VT.csv",
    "CM_2_3_d2_4": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s2_3_d2_4_20250623_151153.1.csv",
    "CM_2_4_sub2_3": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s2_4_sub2_3_20250623_143348.1_VT.csv",
    "CM_3_5_d3_6": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s3_5_d3_6_20250623_160001.csv",
    "CM_3_6_sub3_5": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s3_6_sub3_5_20250623_174348.csv",
    "CM_4_7_d4_8": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s4_7_d4_8_20250623_193718.csv",
    "CM_4_8_sub4_7": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s4_8_sub4_7_20250623_182649.1_VT.csv",
}

# ======================================================
# Summary check
# ======================================================

def summarize_paths(resp_dict, boris_dict, label):
    resp_set = set(resp_dict.keys())
    boris_set = set(boris_dict.keys())
    missing = resp_set - boris_set
    print(f"\n {label} summary")
    print(f"Resp files: {len(resp_set)} | BORIS files: {len(boris_set)} | No BORIS: {len(missing)}")
    if missing:
        print("Missing trials:", ", ".join(sorted(missing)))

# ======================================================
# Organized summaries by experiment type
# ======================================================

# --- Split out subsets for clarity ---
resp_paths_valence = {k: v for k, v in resp_paths.items() if k.startswith(("RI1", "RI2"))}
resp_paths_blri = {k: v for k, v in resp_paths.items() if k.startswith("BLRI")}
resp_paths_cm_only = {k: v for k, v in resp_paths_cm.items() if k.startswith("CM")}
resp_paths_bl_only = {k: v for k, v in resp_paths_bl.items() if k.startswith("BL")}

# Filter BORIS dicts by matching prefixes
boris_paths_valence = {k: v for k, v in boris_paths.items() if k.startswith(("RI1", "RI2"))}
boris_paths_blri = {k: v for k, v in boris_paths.items() if k.startswith("BLRI")}
boris_paths_cm_only = {k: v for k, v in boris_paths_cm.items() if k.startswith("CM")}
boris_paths_bl_only = {k: v for k, v in boris_paths_cm.items() if k.startswith("BL")}

summarize_paths(resp_paths_valence, boris_paths_valence, "Valence (RI1/RI2)")
summarize_paths(resp_paths_blri, boris_paths_blri, "Pre-Valence Baseline (BLRI)")
summarize_paths(resp_paths_cm_only, boris_paths_cm_only, "Cagemate (CM)")
summarize_paths(resp_paths_bl_only, boris_paths_bl_only, "Pre-Cagemate Baseline (BL)")


# ======================================================
# Combine all for unified processing later
# ======================================================
all_resp_paths = {**resp_paths_valence, **resp_paths_blri, **resp_paths_cm_only, **resp_paths_bl_only}
all_boris_paths = {**boris_paths, **boris_paths_cm}

print(f"\n Combined respiration files total: {len(all_resp_paths)}")
print(f" Combined BORIS files total: {len(all_boris_paths)}")

missing_boris = [k for k in all_resp_paths if k not in all_boris_paths]
print(f" Respiration-only trials (no BORIS): {len(missing_boris)}")
if missing_boris:
    print(', '.join(sorted(missing_boris)))


 Valence (RI1/RI2) summary
Resp files: 16 | BORIS files: 15 | No BORIS: 1
Missing trials: RI1_3_5

 Pre-Valence Baseline (BLRI) summary
Resp files: 7 | BORIS files: 0 | No BORIS: 7
Missing trials: BLRI_1_1, BLRI_1_2, BLRI_2_3, BLRI_2_4, BLRI_3_6, BLRI_4_7, BLRI_4_8

 Cagemate (CM) summary
Resp files: 8 | BORIS files: 8 | No BORIS: 0

 Pre-Cagemate Baseline (BL) summary
Resp files: 8 | BORIS files: 0 | No BORIS: 8
Missing trials: BL_1_1_d1_2, BL_1_2_sub1_1, BL_2_3_d2_4, BL_2_4_sub2_3, BL_3_5_d3_6, BL_3_6_sub3_5, BL_4_7_d4_8, BL_4_8_sub4_7

 Combined respiration files total: 39
 Combined BORIS files total: 23
 Respiration-only trials (no BORIS): 16
BLRI_1_1, BLRI_1_2, BLRI_2_3, BLRI_2_4, BLRI_3_6, BLRI_4_7, BLRI_4_8, BL_1_1_d1_2, BL_1_2_sub1_1, BL_2_3_d2_4, BL_2_4_sub2_3, BL_3_5_d3_6, BL_3_6_sub3_5, BL_4_7_d4_8, BL_4_8_sub4_7, RI1_3_5


In [4]:
import pandas as pd

def load_clean_boris(csv_path):
    """
    Load a BORIS CSV file and standardize columns for behavior alignment.

    Returns
    -------
    df : DataFrame
        Columns: ['Behavior', 'Subject', 'Start', 'Stop', 'Duration']
        Keeps only 'subject' rows and social behaviors.
    """
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"⚠️ Could not load {csv_path}: {e}")
        return pd.DataFrame()

    # --- Detect and rename possible column variants ---
    rename_map = {
        "Behavior": "Behavior",
        "Subject": "Subject",
        "Start (s)": "Start",
        "Stop (s)": "Stop",
        "Duration (s)": "Duration",
        "Start": "Start",
        "Stop": "Stop",
        "Duration": "Duration"
    }
    df = df.rename(columns=rename_map)

    # --- Keep essential columns only ---
    keep_cols = [c for c in ["Behavior", "Subject", "Start", "Stop", "Duration"] if c in df.columns]
    df = df[keep_cols].copy()

    # --- Clean ---
    df = df.dropna(subset=["Behavior", "Subject", "Start", "Stop"])
    df["Behavior"] = df["Behavior"].str.lower().str.strip()
    df["Subject"] = df["Subject"].str.lower().str.strip()

    # --- Keep only subject-initiated behaviors ---
    df = df[df["Subject"] == "subject"]

    # --- Focus on relevant social behaviors ---
    behaviors_keep = ["facial sniffing", "body sniffing", "anogenital sniffing"]
    df = df[df["Behavior"].isin(behaviors_keep)]

    # --- Sort chronologically ---
    df = df.sort_values("Start").reset_index(drop=True)

    return df

In [5]:
boris_path = r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_2_p5_1_nRB6_20250622_170742.1.csv"
boris_df = load_clean_boris(boris_path)

print(boris_df.head())

              Behavior  Subject   Start    Stop  Duration
0        body sniffing  subject  16.155  18.200     2.045
1        body sniffing  subject  19.300  21.767     2.467
2        body sniffing  subject  23.833  25.100     1.267
3  anogenital sniffing  subject  25.567  26.867     1.300
4        body sniffing  subject  27.233  27.833     0.600


In [6]:
def load_clean_resp_signal(h5_file, target_rate=100):
    """
    Loads, filters, downsamples respiration from .h5, returns cleaned signal, time vector, and metadata.
    """
    try:
        with h5py.File(h5_file, 'r') as f:
            resp = f['resp'][:].flatten()

            # Load metadata if available
            metadata = {}
            if 'resp_metadata' in f:
                metadata.update(dict(f['resp_metadata'].attrs))
            if 'ekg_metadata' in f:
                metadata.update(dict(f['ekg_metadata'].attrs))
            if 'metadata' in f:
                metadata.update(dict(f['metadata'].attrs))

            # Estimate sampling frequency
            if 'sampling_frequency' in metadata:
                fs = metadata['sampling_frequency']
            else:
                duration_sec = metadata.get('duration_sec', None)
                fs = len(resp) / duration_sec if duration_sec else 20000.0

            duration_sec = metadata.get('duration_sec', len(resp) / fs)

    except Exception as e:
        print(f"Error loading {h5_file}: {e}")
        return None, None, None, None

    # Pre-filter before downsampling
    nyquist = fs / 2
    norm_cutoff = (target_rate / 2) / nyquist
    b, a = butter(N=4, Wn=norm_cutoff, btype='low')
    filtered_resp = filtfilt(b, a, resp)

    # Downsample
    downsample_factor = int(fs // target_rate)
    downsampled = resample_poly(filtered_resp, up=1, down=downsample_factor)

    # Bandpass filter with neurokit
    rsp_cleaned = nk.signal_filter(
        downsampled,
        lowcut=0.1,
        highcut=20,
        method="butterworth",
        sampling_rate=target_rate,
        order=2
    )

    # Generate matching time vector
    time_vector = np.arange(len(rsp_cleaned)) / target_rate

    return rsp_cleaned, time_vector, target_rate, metadata


In [7]:
def get_sniff_respiratory_rate(signal, time, sniff_start, sniff_end, sampling_rate=100):
    # mask the signal window
    sniff_mask = (time >= sniff_start) & (time < sniff_end)
    signal_sniff = signal[sniff_mask]
    time_sniff = time[sniff_mask]

    # detect peaks with a tighter minimum distance 12 Hz max rate)
    peaks, _ = find_peaks(signal_sniff, distance=sampling_rate * 0.0833)  # 0.0833 s = 12 Hz
    peak_times = time_sniff[peaks]

    # need at least 2 peaks to compute IBI
    if len(peak_times) < 2:
        return np.nan

    # compute IBI
    ibi = np.diff(peak_times)  # seconds

    # instantaneous rate
    inst_rate = 1.0 / ibi      # Hz

    # return average instantaneous rate (one number, same format as old code)
    avg_rate = np.mean(inst_rate)

    return avg_rate

In [8]:
# --- Pick one session to test ---
trial_key = "CM_2_4_sub2_3"
resp_path = resp_paths_cm[trial_key]

# --- Load respiration ---
resp_signal, time, fs, meta = load_clean_resp_signal(resp_path)

# --- Compute session-wide respiration rate ---
sniff_start = time[0]
sniff_end = time[-1]

avg_rate = get_sniff_respiratory_rate(
    signal=resp_signal,
    time=time,
    sniff_start=sniff_start,
    sniff_end=sniff_end,
    sampling_rate=fs
)

# --- Extract identifiers automatically ---
condition = trial_key.split("_")[0]              # e.g., "CM"
subject_id = "_".join(trial_key.split("_")[1:3]) # e.g., "2_4"

# --- Print summary ---
print(f"Session: {condition} | Subject: {subject_id}")
print(f"Average respiration rate: {avg_rate:.2f} Hz")
print(f"Sampling rate: {fs:.0f} Hz | Duration: {time[-1]:.1f} sec")

Session: CM | Subject: 2_4
Average respiration rate: 7.49 Hz
Sampling rate: 100 Hz | Duration: 622.0 sec


# `process_all_trials()` — Function Overview

This function iterates through all respiration and BORIS files, computes respiration metrics at both the session level and bout level, handles missing BORIS sessions, attaches metadata (subject, condition, rank), and returns one combined DataFrame for downstream analysis.

---

## 📌 Purpose

`process_all_trials()` creates a unified dataset by:

- Loading respiration signals for each trial  
- Computing **session-wide respiratory rate**  
- Loading BORIS behavior windows (if available)  
- Computing **bout-level respiration rate** for each behavior  
- Filtering short/unreliable behavior bouts  
- Preserving respiration-only baseline sessions  
- Adding metadata (Subject, Condition, Rank, Trial, Type)  
- Concatenating all trials into one master DataFrame  

---

## 📥 Inputs

- **resp_paths**: dict mapping trial name → respiration `.h5` file  
- **boris_paths**: dict mapping trial name → BORIS `.csv` file  
- **rank_map**: maps subject ID → social rank label  
- **duration_threshold**: minimum bout duration (in seconds) to keep  

---

## 🧠 What the Function Does Step-by-Step

### 1. Loop through all trials  
For each trial in `resp_paths`, load its respiration file.

### 2. Load respiration signal  
Uses `load_clean_resp_signal()` to extract:
- filtered respiration signal  
- timestamps  
- sampling rate  
- metadata  

If loading fails, the trial is skipped.

### 3. Compute session-wide respiratory rate  
Calls `get_sniff_respiratory_rate()` across the entire recording  
(`time[0]` → `time[-1]`).

Stored later as `SessionRate`.

### 4. Parse trial metadata  
From the trial name (e.g., `"RI1_3_6"`):
- Subject = `"3_6"`  
- Condition = `"RI1"`  
- Rank = lookup via `rank_map`  

### 5. Load BORIS behavior (if available)  
If the trial exists in `boris_paths`, load it using `load_clean_boris()`.  
If not, create an empty DataFrame.

### 6. Handle missing BORIS sessions  
If BORIS is missing (e.g., baseline trials):
- Save a **single row** containing only the session respiratory rate  
- Tag it with `Type = "Baseline"`  

This ensures baseline sessions are not lost.

### 7. Compute bout-level respiration rate  
For each BORIS row:

`MeanRate = get_sniff_respiratory_rate(Start → Stop)`

This computes sniff frequency **within that specific behavior window**.

### 8. Filter out short bouts  
Remove windows where `Duration < duration_threshold`.  
Avoids unstable rate estimates from very short behavior bouts.

### 9. Append metadata  
Each row receives:
- Trial  
- Subject  
- Condition  
- Rank  
- SessionRate  
- Type = `"Interaction"`  

### 10. Combine all trials into one dataset  
Concatenates all processed DataFrames into `master_df`.

Returns the final master DataFrame.

---

## 📊 Output Columns

The final DataFrame includes:

| Column        | Description |
|---------------|-------------|
| Behavior      | Behavior label from BORIS |
| Start / Stop  | Window timestamps |
| Duration      | Length of behavior bout |
| MeanRate      | Respiration rate during the bout |
| SessionRate   | Respiration rate across the entire session |
| Subject       | Mouse ID (e.g., `3_6`) |
| Condition     | Trial condition (RI1, RI2, BLRI) |
| Rank          | Dominant / Subordinate |
| Trial         | Trial identifier |
| Type          | `"Interaction"` or `"Baseline"` |

---

## ✅ Summary

`process_all_trials()` converts raw respiration and BORIS data into a clean, analysis-ready dataset that includes:

- Session-level physiological features  
- Bout-level respiration features  
- Social context metadata  
- Rank and subject identifiers  
- Handling of missing BORIS recordings  

This serves as the foundation for decoding models, ANOVAs, and visualization of respiration-behavior relationships.


In [9]:
def process_all_trials(resp_paths, boris_paths, rank_map, duration_threshold=0.5):
    """
    Process respiration and BORIS behavioral data across multiple trials.

    This function iterates through all respiration files listed in `resp_paths`,
    loads the cleaned respiration signal, computes a session-wide respiratory
    rate, integrates BORIS behavior annotations when available, computes 
    bout-level respiratory rates, filters short bouts, attaches metadata, 
    handles sessions without BORIS, and returns a combined master DataFrame.

    -------------------------------------------------------------------------
    Workflow Summary
    -------------------------------------------------------------------------
    For each trial:
        1. Load the respiration H5 file
           - Extracts the filtered respiration waveform, timestamps, and 
             sampling rate using `load_clean_resp_signal()`.

        2. Compute the session-wide respiratory rate
           - Uses `get_sniff_respiratory_rate()` across the full session 
             (time[0] → time[-1]).
           - Stored as `SessionRate` for every row in that trial.

        3. Parse trial metadata
           - Extract Subject and Condition from the trial name.
           - Look up Rank using `rank_map`.

        4. Load BORIS behavioral annotations (if available)
           - If no BORIS file exists, creates a single-row respiration-only 
             entry for that session and tags it as Type = "Baseline".

        5. Compute bout-level respiration rate (if BORIS exists)
           - For each behavior window (Start → Stop), compute sniff frequency 
             using `get_sniff_respiratory_rate()`.
           - Stored as `MeanRate`.

        6. Filter short bouts
           - Removes BORIS windows with Duration < duration_threshold (default 0.5s).

        7. Attach metadata
           - Adds Trial, Subject, Condition, Rank, SessionRate.
           - Tags rows with `Type = "Interaction"`.

    -------------------------------------------------------------------------
    Output Structure
    -------------------------------------------------------------------------
    Returns a single concatenated pandas DataFrame containing:

        Behavior       - BORIS behavior label (or NaN for baseline)
        Start, Stop    - Time windows for behavior
        Duration       - Bout duration in seconds
        MeanRate       - Respiratory rate during each bout (NaN if baseline)
        SessionRate    - Respiratory rate across the entire session
        Trial          - Trial name (e.g., "RI1_3_6")
        Subject        - Mouse ID extracted from trial name
        Condition      - Trial condition (RI1, RI2, BLRI, etc.)
        Rank           - Dominant/Subordinate based on rank_map
        Type           - "Interaction" or "Baseline"

    -------------------------------------------------------------------------
    Parameters
    -------------------------------------------------------------------------
    resp_paths : dict
        Mapping from trial names to respiration .h5 file paths.
    boris_paths : dict
        Mapping from trial names to BORIS .csv file paths.
    rank_map : dict
        Mapping from subject IDs (e.g., "3_6") to rank labels.
    duration_threshold : float, optional
        Minimum bout duration (in seconds) required to include a behavior window.
        Defaults to 0.5 seconds.

    -------------------------------------------------------------------------
    Returns
    -------------------------------------------------------------------------
    master_df : pandas.DataFrame
        Combined dataset containing bout-level and session-level respiration
        features, full trial metadata, and baseline sessions when BORIS is missing.

    """

    all_trials = []

    for trial, h5_path in resp_paths.items():
        print(f"Processing {trial}...")

        # --- Load respiration ---
        signal, time, fs, meta = load_clean_resp_signal(h5_path)
        if signal is None:
            print(f"Resp load failed for {trial}")
            continue

        # --- Compute session-wide rate ---
        session_rate = get_sniff_respiratory_rate(
            signal=signal,
            time=time,
            sniff_start=time[0],
            sniff_end=time[-1],
            sampling_rate=fs
        )
        print(f"Session-wide respiratory rate for {trial}: {session_rate:.3f} Hz")

        # --- Determine subject, condition, rank ---
        subj = "_".join(trial.split("_")[1:3])
        condition = trial.split("_")[0]
        rank = rank_map.get(subj, np.nan)

        # --- Try loading BORIS ---
        if trial in boris_paths:
            boris_df = load_clean_boris(boris_paths[trial])
        else:
            boris_df = pd.DataFrame()  # missing BORIS entirely

        # --- Handle missing BORIS (resp-only) ---
        if boris_df.empty:
            print(f"No BORIS data for {trial} — saving respiration-only session")
            all_trials.append(pd.DataFrame([{
                "Behavior": np.nan,
                "Start": np.nan,
                "Stop": np.nan,
                "Duration": np.nan,
                "MeanRate": np.nan,
                "Trial": trial,
                "Subject": subj,
                "Condition": condition,
                "Rank": rank,
                "SessionRate": session_rate,
                "Type": "Baseline"
            }]))
            continue

        # --- Compute respiration rate per behavior window ---
        boris_df["MeanRate"] = boris_df.apply(
            lambda row: get_sniff_respiratory_rate(
                signal, time, row["Start"], row["Stop"], sampling_rate=fs
            ),
            axis=1
        )

        # --- Filter short bouts ---
        pre_len = len(boris_df)
        boris_df = boris_df[boris_df["Duration"] >= duration_threshold].copy()
        post_len = len(boris_df)
        if pre_len != post_len:
            print(f"   → Filtered {pre_len - post_len} short bouts (<{duration_threshold}s)")

        # --- Add metadata ---
        boris_df["Trial"] = trial
        boris_df["Subject"] = subj
        boris_df["Condition"] = condition
        boris_df["Rank"] = rank
        boris_df["SessionRate"] = session_rate
        boris_df["Type"] = "Interaction"

        all_trials.append(boris_df)

    # --- Combine everything ---
    if not all_trials:
        print("No valid trials processed.")
        return pd.DataFrame()

    master_df = pd.concat(all_trials, ignore_index=True)
    print(f"Combined {len(master_df)} behavior windows across {len(all_trials)} trials.")
    print("Unique session types:", master_df["Type"].unique())
    return master_df

In [10]:
# ======================================================
# Process all trial sets (Valence, Cagemate, Baseline)
# ======================================================

print("\n==============================")
print("Processing Valence (RI1/RI2/BLRI)")
print("==============================")
master_df_valence = process_all_trials(resp_paths, boris_paths, rank_map)

print("\n==============================")
print("Processing Cagemate (CM)")
print("==============================")
master_df_cm = process_all_trials(resp_paths_cm, boris_paths_cm, rank_map)

print("\n==============================")
print("Processing Pre-Cagemate Baseline (BL)")
print("==============================")
master_df_bl = process_all_trials(resp_paths_bl, {}, rank_map)

# --- Combine all into one master DataFrame ---
print("\n==============================")
print("Combining all master DataFrames")
print("==============================")

master_df = pd.concat(
    [master_df_valence, master_df_cm, master_df_bl],
    ignore_index=True
)

print(f" Final combined dataset: {len(master_df)} rows total")
print("Unique conditions:", master_df['Condition'].unique())
print("Unique ranks:", master_df['Rank'].unique())



Processing Valence (RI1/RI2/BLRI)
Processing RI1_3_6...
Session-wide respiratory rate for RI1_3_6: 8.079 Hz
   → Filtered 4 short bouts (<0.5s)
Processing RI2_3_6...
Session-wide respiratory rate for RI2_3_6: 6.513 Hz
   → Filtered 12 short bouts (<0.5s)
Processing RI1_4_7...
Session-wide respiratory rate for RI1_4_7: 8.339 Hz
   → Filtered 3 short bouts (<0.5s)
Processing RI2_4_7...
Session-wide respiratory rate for RI2_4_7: 7.029 Hz
   → Filtered 1 short bouts (<0.5s)
Processing RI1_2_3...
Session-wide respiratory rate for RI1_2_3: 8.207 Hz
Processing RI2_2_3...
Session-wide respiratory rate for RI2_2_3: 7.352 Hz
Processing RI1_4_8...
Session-wide respiratory rate for RI1_4_8: 8.211 Hz
   → Filtered 8 short bouts (<0.5s)
Processing RI2_4_8...
Session-wide respiratory rate for RI2_4_8: 6.226 Hz
   → Filtered 1 short bouts (<0.5s)
Processing RI1_1_1...
Session-wide respiratory rate for RI1_1_1: 8.204 Hz
   → Filtered 4 short bouts (<0.5s)
Processing RI2_1_1...
Session-wide respiratory

In [11]:
master_df

,Behavior,Subject,Start,Stop,Duration,MeanRate,Trial,Condition,Rank,SessionRate,Type
0,facial sniffing,3_6,11.966,13.000,1.034,8.814815,RI1_3_6,RI1,Dominant,8.078920,Interaction
1,anogenital sniffing,3_6,14.483,16.690,2.207,9.704521,RI1_3_6,RI1,Dominant,8.078920,Interaction
2,anogenital sniffing,3_6,25.034,27.552,2.518,9.152463,RI1_3_6,RI1,Dominant,8.078920,Interaction
3,body sniffing,3_6,27.793,28.448,0.655,10.040404,RI1_3_6,RI1,Dominant,8.078920,Interaction
4,anogenital sniffing,3_6,28.862,29.861,0.999,9.180884,RI1_3_6,RI1,Dominant,8.078920,Interaction
...,...,...,...,...,...,...,...,...,...,...,...
476,NaN,2_4,NaN,NaN,NaN,NaN,BL_2_4_sub2_3,BL,Dominant,8.471447,Baseline
477,NaN,3_5,NaN,NaN,NaN,NaN,BL_3_5_d3_6,BL,Subordinate,8.381803,Baseline
478,NaN,3_6,NaN,NaN,NaN,NaN,BL_3_6_sub3_5,BL,Dominant,6.000826,Baseline
479,NaN,4_7,NaN,NaN,NaN,NaN,BL_4_7_d4_8,BL,Subordinate,8.282217,Baseline


In [12]:
valence_df = master_df[
    (master_df["Type"] == "Interaction") &
    (master_df["Condition"].isin(["RI1", "RI2"]))
].copy()

# Binary valence label (adjust mapping if needed)
valence_df["ValenceLabel"] = valence_df["Condition"].map({
    "RI1": 0,   # e.g. affiliative
    "RI2": 1    # e.g. aversive
})

features = ["MeanRate"]   # keep minimal for interpretability
X = valence_df[features].values
y = valence_df["ValenceLabel"].values
groups = valence_df["Subject"].values


In [26]:
valence_df


,Behavior,Subject,Start,Stop,Duration,MeanRate,Trial,Condition,Rank,SessionRate,Type,ValenceLabel
0,facial sniffing,3_6,11.966,13.000,1.034,8.814815,RI1_3_6,RI1,Dominant,8.078920,Interaction,0
1,anogenital sniffing,3_6,14.483,16.690,2.207,9.704521,RI1_3_6,RI1,Dominant,8.078920,Interaction,0
2,anogenital sniffing,3_6,25.034,27.552,2.518,9.152463,RI1_3_6,RI1,Dominant,8.078920,Interaction,0
3,body sniffing,3_6,27.793,28.448,0.655,10.040404,RI1_3_6,RI1,Dominant,8.078920,Interaction,0
4,anogenital sniffing,3_6,28.862,29.861,0.999,9.180884,RI1_3_6,RI1,Dominant,8.078920,Interaction,0
...,...,...,...,...,...,...,...,...,...,...,...,...
302,body sniffing,2_4,9.000,12.600,3.600,8.934506,RI2_2_4,RI2,Dominant,6.749941,Interaction,1
303,body sniffing,2_4,18.000,20.133,2.133,9.727273,RI2_2_4,RI2,Dominant,6.749941,Interaction,1
304,body sniffing,2_4,26.333,30.000,3.667,9.332629,RI2_2_4,RI2,Dominant,6.749941,Interaction,1
305,facial sniffing,2_4,77.400,79.667,2.267,9.437951,RI2_2_4,RI2,Dominant,6.749941,Interaction,1


In [18]:
gkf = GroupKFold(n_splits=len(np.unique(groups)))

valence_auc = []

for train_idx, test_idx in gkf.split(X, y, groups):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(solver="liblinear"))
    ])

    pipe.fit(X[train_idx], y[train_idx])
    probs = pipe.predict_proba(X[test_idx])[:, 1]
    auc = roc_auc_score(y[test_idx], probs)
    valence_auc.append(auc)

print(f"Valence decoding AUC (LOSOCV): {np.mean(valence_auc):.3f} ± {np.std(valence_auc):.3f}")


Valence decoding AUC (LOSOCV): nan ± nan


c:\Users\sjs93\anaconda3\envs\biopipeline-env\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\sjs93\anaconda3\envs\biopipeline-env\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [19]:
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np

gkf = GroupKFold(n_splits=len(np.unique(groups)))

valence_auc = []
skipped_subjects = []

for train_idx, test_idx in gkf.split(X, y, groups):

    # --- Check if both classes are present in test set ---
    if len(np.unique(y[test_idx])) < 2:
        skipped_subjects.append(groups[test_idx][0])
        continue

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(solver="liblinear"))
    ])

    pipe.fit(X[train_idx], y[train_idx])
    probs = pipe.predict_proba(X[test_idx])[:, 1]
    auc = roc_auc_score(y[test_idx], probs)
    valence_auc.append(auc)

print(f"Valence decoding AUC (LOSOCV): {np.mean(valence_auc):.3f} ± {np.std(valence_auc):.3f}")
print(f"Skipped subjects (single-class test set): {np.unique(skipped_subjects)}")


Valence decoding AUC (LOSOCV): 0.614 ± 0.249
Skipped subjects (single-class test set): ['3_5' '4_8']


In [22]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(solver="liblinear"))
])

pipe.fit(X, y)   # all valence bouts


,steps,"[('scaler', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [23]:
rank_df["ValenceProb"] = pipe.predict_proba(X_rank)[:, 1]


In [24]:
subject_rank_df = (
    rank_df
    .groupby(["Subject", "Rank"])["ValenceProb"]
    .mean()
    .reset_index()
)

subject_rank_df["RankLabel"] = subject_rank_df["Rank"].map({
    "Subordinate": 0,
    "Dominant": 1
})


In [25]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(
    subject_rank_df["RankLabel"],
    subject_rank_df["ValenceProb"]
)

print(f"Valence → Rank transfer AUC (subject-level): {auc:.3f}")


Valence → Rank transfer AUC (subject-level): 0.312
